# Лабораторная работа 1. Линейная регрессия и факторный анализ

Датасет: **Medical Cost Personal Dataset** (`insurance.csv`).

Цель работы - предсказать медицинские расходы пациента (`charges`) и сравнить линейную, гребневую и лассо-регрессию до и после устранения мультиколлинеарности методом главных компонент.

Запускайте ячейки сверху вниз.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
DATA_PATH = DATA_DIR / "insurance.csv"
DATA_URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5
VARIANCE_THRESHOLD = 0.95

FEATURES = [
    "age",
    "bmi",
    "children",
    "sex_male",
    "smoker_yes",
    "region_northwest",
    "region_southeast",
    "region_southwest",
]
TARGET = "charges"

LABELS = {
    "charges": "Медицинские расходы",
    "age": "Возраст",
    "bmi": "Индекс массы тела",
    "children": "Количество детей",
    "sex_male": "Пол: мужчина",
    "smoker_yes": "Курит",
    "region_northwest": "Регион: northwest",
    "region_southeast": "Регион: southeast",
    "region_southwest": "Регион: southwest",
}
MODEL_TITLES = {"linear": "Линейная", "ridge": "Гребневая", "lasso": "Лассо"}
FEATURE_SET_TITLES = {"original": "Исходные признаки", "pca": "Главные компоненты"}

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", font="DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150

## 1. Загрузка датасета

Датасет скачивается в папку `data`. Если файл уже есть, повторная загрузка не выполняется.

In [ ]:
if not DATA_PATH.exists() or DATA_PATH.stat().st_size == 0:
    print(f"Скачивание датасета: {DATA_URL}")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
else:
    print(f"Датасет уже скачан: {DATA_PATH}")

raw = pd.read_csv(DATA_PATH)
print(f"Размер исходного датасета: {raw.shape[0]} строк, {raw.shape[1]} столбцов")
print("Пропущенные значения:")
display(raw.isna().sum().to_frame("Пропусков"))
display(raw.head())

### Результат загрузки

| Показатель | Значение |
| --- | ---: |
| Наблюдений | 1338 |
| Столбцов | 7 |
| Целевая переменная | `charges` |
| Пропуски | 0 |

Файл датасета сохранен как `data/insurance.csv`.

## 2. Первичный анализ

Проверим описательные статистики и подготовим категориальные признаки для регрессионных моделей.

In [ ]:
data = raw.dropna().reset_index(drop=True)
print(f"Удалено строк с пропусками: {len(raw) - len(data)}")
print(f"Размер после очистки: {data.shape[0]} строк")

data = pd.get_dummies(data, columns=["sex", "smoker", "region"], drop_first=True, dtype=float)
stats = data[FEATURES + [TARGET]].describe().T
stats.to_csv(OUTPUT_DIR / "describe.csv", encoding="utf-8-sig")
display(stats.round(2))

### Результат первичного анализа

После очистки осталось 1338 наблюдений. Категориальные признаки `sex`, `smoker`, `region` закодированы методом one-hot с удалением первой категории. В модель входят 8 признаков.

## 3. Визуализация признаков и целевой переменной

Построим распределения признаков и посмотрим связь медицинских расходов с BMI и курением.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
plot_columns = ["age", "bmi", "children", "charges", "sex_male", "smoker_yes", "region_southeast", "region_southwest"]
for ax, column in zip(axes.ravel(), plot_columns):
    sns.histplot(data[column], bins=25, ax=ax, color="#4C78A8")
    ax.set_title(LABELS.get(column, column))
    ax.set_xlabel("")
    ax.set_ylabel("Частота")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_distributions.png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=raw, x="bmi", y="charges", hue="smoker", alpha=0.7)
plt.title("Расходы, BMI и курение")
plt.xlabel("BMI")
plt.ylabel("Медицинские расходы")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_target_vs_bmi_smoker.png", bbox_inches="tight")
plt.show()

### Результат визуализации

![Распределения признаков и целевой переменной](output/01_distributions.png)

![Связь расходов с BMI и курением](output/02_target_vs_bmi_smoker.png)

`charges` имеет правостороннюю асимметрию. Курящие пациенты заметно чаще имеют высокие медицинские расходы.

## 4. Корреляционная матрица

Посмотрим парные корреляции признаков и целевой переменной.

In [ ]:
corr = data[FEATURES + [TARGET]].corr()
corr.to_csv(OUTPUT_DIR / "correlation.csv", encoding="utf-8-sig")

plt.figure(figsize=(10, 8))
sns.heatmap(corr.rename(index=LABELS, columns=LABELS), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляционная матрица")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_correlation.png", bbox_inches="tight")
plt.show()

target_corr = corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
display(target_corr.to_frame("corr_with_target").round(3))

### Результат корреляционного анализа

![Корреляционная матрица](output/03_correlation.png)

Сильнее всего с медицинскими расходами связан признак `smoker_yes`: корреляция около 0.79. Также положительная связь есть у возраста и BMI.

## 5. Мультиколлинеарность: расчет VIF

VIF показывает, насколько каждый признак объясняется остальными. Значения выше 5 считают заметной мультиколлинеарностью, выше 10 - сильной.

In [ ]:
matrix = add_constant(data[FEATURES].astype(float))
vif = pd.DataFrame({
    "feature": FEATURES,
    "label": [LABELS[name] for name in FEATURES],
    "VIF": [variance_inflation_factor(matrix.values, i + 1) for i in range(len(FEATURES))],
}).sort_values("VIF", ascending=False)
vif.to_csv(OUTPUT_DIR / "vif_before.csv", index=False, encoding="utf-8-sig")
display(vif.round(2))

plt.figure(figsize=(8, 4.8))
sns.barplot(vif, x="VIF", y="label", color="#F58518")
plt.axvline(5, color="red", linestyle="--", label="VIF = 5")
plt.axvline(10, color="purple", linestyle="--", label="VIF = 10")
plt.xlabel("VIF")
plt.ylabel("")
plt.title("VIF до PCA")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_vif.png", bbox_inches="tight")
plt.show()

### Результат VIF

| Признак | VIF |
| --- | ---: |
| Регион: southeast | 1.65 |
| Регион: southwest | 1.53 |
| Регион: northwest | 1.52 |
| Индекс массы тела | 1.11 |
| Возраст | 1.02 |
| Курит | 1.01 |
| Пол: мужчина | 1.01 |
| Количество детей | 1.00 |

![VIF до PCA](output/04_vif.png)

Критической мультиколлинеарности нет: все значения VIF ниже 5.

## 6. Обучение, метрики и кросс-валидация

Обучим линейную, гребневую и лассо-регрессию. Для Ridge и Lasso параметр `alpha` подбирается по сетке с 5-fold кросс-валидацией.

In [ ]:
def mean_absolute_percentage_error_percent(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


def evaluate_predictions(y_true, y_pred):
    return {
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error_percent(y_true, y_pred),
    }


def train_models(x_train, x_test, y_train, y_test, feature_set, feature_names):
    alpha_grid = np.logspace(-3, 3, 50)
    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {"rmse": "neg_root_mean_squared_error", "r2": "r2", "mape": "neg_mean_absolute_percentage_error"}
    models = {
        "linear": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
        "ridge": GridSearchCV(Pipeline([("scaler", StandardScaler()), ("model", Ridge())]), {"model__alpha": alpha_grid}, cv=cv, scoring="neg_root_mean_squared_error"),
        "lasso": GridSearchCV(Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=50_000, random_state=RANDOM_STATE))]), {"model__alpha": alpha_grid}, cv=cv, scoring="neg_root_mean_squared_error"),
    }
    rows, predictions, fitted, coefficients = [], {}, {}, []
    for model_name, estimator in models.items():
        cv_result = cross_validate(estimator, x_train, y_train, cv=cv, scoring=scoring)
        estimator.fit(x_train, y_train)
        best_estimator = estimator.best_estimator_ if isinstance(estimator, GridSearchCV) else estimator
        y_pred = best_estimator.predict(x_test)
        predictions[model_name] = y_pred
        fitted[model_name] = best_estimator
        alpha = estimator.best_params_["model__alpha"] if isinstance(estimator, GridSearchCV) else np.nan
        rows.append({
            "feature_set": feature_set,
            "features_title": FEATURE_SET_TITLES[feature_set],
            "model": model_name,
            "model_title": MODEL_TITLES[model_name],
            "alpha": alpha,
            "CV_RMSE": -cv_result["test_rmse"].mean(),
            "CV_R2": cv_result["test_r2"].mean(),
            "CV_MAPE": -cv_result["test_mape"].mean() * 100,
            **evaluate_predictions(y_test, y_pred),
        })
        model = best_estimator.named_steps["model"]
        for name, coef in zip(feature_names, model.coef_):
            coefficients.append({"feature_set": feature_set, "model": model_name, "model_title": MODEL_TITLES[model_name], "feature": name, "label": LABELS.get(name, name), "coefficient": coef})
    return pd.DataFrame(rows), predictions, fitted, pd.DataFrame(coefficients)

In [ ]:
x = data[FEATURES]
y = data[TARGET]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"Обучающая выборка: {x_train.shape[0]} строк")
print(f"Тестовая выборка: {x_test.shape[0]} строк")

before, pred_before, fitted_before, coef_before = train_models(x_train, x_test, y_train, y_test, "original", FEATURES)
display(before.round(3))
display(coef_before.pivot(index="label", columns="model_title", values="coefficient").round(2))
coef_before.to_csv(OUTPUT_DIR / "coefficients_original.csv", index=False, encoding="utf-8-sig")

### Результат обучения на исходных признаках

| Модель | alpha | CV RMSE | CV R² | CV MAPE, % | RMSE | R² | MAPE, % |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Линейная | - | 6123.35 | 0.739 | 42.41 | 5796.28 | 0.784 | 46.89 |
| Гребневая | 2.0236 | 6123.34 | 0.739 | 42.52 | 5797.71 | 0.783 | 46.97 |
| Лассо | 104.8113 | 6132.23 | 0.738 | 42.84 | 5837.96 | 0.780 | 48.20 |

По RMSE лучшая модель на исходных признаках - линейная регрессия.

## 7. PCA: устранение мультиколлинеарности

Перед PCA признаки стандартизируются. Число компонент выбирается по правилу сохранения не менее 95% дисперсии.

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

pca_full = PCA().fit(x_train_scaled)
explained = pca_full.explained_variance_ratio_
cumulative = explained.cumsum()
n_components = int(np.argmax(cumulative >= VARIANCE_THRESHOLD) + 1)
print(f"Компонент для сохранения {VARIANCE_THRESHOLD:.0%} дисперсии: {n_components}")

pca_variance = pd.DataFrame({"component": [f"PC{i}" for i in range(1, len(explained) + 1)], "eigenvalue": pca_full.explained_variance_, "explained_variance_ratio": explained, "cumulative_variance": cumulative})
pca_variance.to_csv(OUTPUT_DIR / "pca_variance.csv", index=False, encoding="utf-8-sig")
display(pca_variance.round(3))

plt.figure(figsize=(7, 4.5))
plt.plot(range(1, len(explained) + 1), explained, marker="o", label="Доля дисперсии")
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="s", label="Накопленная доля")
plt.axhline(VARIANCE_THRESHOLD, color="red", linestyle="--", label="95%")
plt.xticks(range(1, len(explained) + 1))
plt.xlabel("Номер компоненты")
plt.ylabel("Доля дисперсии")
plt.title("График каменистой осыпи")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_scree.png", bbox_inches="tight")
plt.show()

pca = PCA(n_components=n_components)
x_train_pca = pca.fit_transform(x_train_scaled)
x_test_pca = pca.transform(x_test_scaled)
pc_names = [f"PC{i}" for i in range(1, n_components + 1)]
pca_loadings = pd.DataFrame(pca.components_.T, index=FEATURES, columns=pc_names)
pca_loadings.to_csv(OUTPUT_DIR / "pca_loadings.csv", encoding="utf-8-sig")
vif_after = pd.DataFrame({"feature": pc_names, "label": pc_names, "VIF": [variance_inflation_factor(x_train_pca, i) for i in range(n_components)]})
vif_after.to_csv(OUTPUT_DIR / "vif_after_pca.csv", index=False, encoding="utf-8-sig")
display(vif_after.round(3))

### Результат PCA

![График каменистой осыпи](output/05_scree.png)

Для сохранения не менее 95% дисперсии достаточно 7 главных компонент. После PCA компоненты ортогональны, поэтому VIF каждой компоненты равен 1.

## 8. Те же модели на главных компонентах

Повторим обучение линейной, гребневой и лассо-регрессии, но теперь в качестве признаков используются главные компоненты.

In [ ]:
after, pred_after, fitted_after, coef_after = train_models(
    pd.DataFrame(x_train_pca, columns=pc_names, index=x_train.index),
    pd.DataFrame(x_test_pca, columns=pc_names, index=x_test.index),
    y_train,
    y_test,
    "pca",
    pc_names,
)
metrics = pd.concat([before, after], ignore_index=True)
metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False, encoding="utf-8-sig")
coef_after.to_csv(OUTPUT_DIR / "coefficients_pca.csv", index=False, encoding="utf-8-sig")
display(metrics.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, metric in zip(axes, ["RMSE", "R2", "MAPE"]):
    sns.barplot(metrics, x="model_title", y=metric, hue="features_title", ax=ax)
    ax.set_xlabel("")
    ax.set_title(metric if metric != "MAPE" else "MAPE, %")
    ax.legend(title="")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_metrics.png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(5.8, 5.8))
plt.scatter(y_test, pred_before["linear"], alpha=0.75, label="Исходные признаки")
plt.scatter(y_test, pred_after["linear"], alpha=0.75, label="PCA")
limits = [min(y_test.min(), pred_before["linear"].min(), pred_after["linear"].min()), max(y_test.max(), pred_before["linear"].max(), pred_after["linear"].max())]
plt.plot(limits, limits, "r--", label="Идеальный прогноз")
plt.xlabel("Фактические расходы")
plt.ylabel("Предсказанные расходы")
plt.title("Факт и прогноз")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "07_pred_vs_actual.png", bbox_inches="tight")
plt.show()

### Результат сравнения моделей

| Тип признаков | Модель | alpha | CV RMSE | CV R² | CV MAPE, % | RMSE | R² | MAPE, % |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Исходные признаки | Линейная | - | 6123.35 | 0.739 | 42.41 | 5796.28 | 0.784 | 46.89 |
| Исходные признаки | Гребневая | 2.0236 | 6123.34 | 0.739 | 42.52 | 5797.71 | 0.783 | 46.97 |
| Исходные признаки | Лассо | 104.8113 | 6132.23 | 0.738 | 42.84 | 5837.96 | 0.780 | 48.20 |
| Главные компоненты | Линейная | - | 6127.98 | 0.739 | 42.34 | 5831.00 | 0.781 | 48.00 |
| Главные компоненты | Гребневая | 1.5264 | 6127.92 | 0.739 | 42.39 | 5831.77 | 0.781 | 48.05 |
| Главные компоненты | Лассо | 25.5955 | 6126.89 | 0.739 | 42.38 | 5833.63 | 0.781 | 47.91 |

![Сравнение метрик](output/06_metrics.png)

![Факт и прогноз](output/07_pred_vs_actual.png)

После PCA качество остается близким, но не улучшается. Исходные признаки удобнее для интерпретации.

## 9. Вывод

Главный фактор медицинских расходов - курение. Также расходы растут с возрастом и BMI. Критической мультиколлинеарности в исходных признаках нет: VIF всех признаков ниже 5.

PCA переводит признаки в ортогональные компоненты и полностью устраняет мультиколлинеарность, но качество прогноза почти не меняется. Для данного датасета линейная модель на исходных признаках остается лучшим и более интерпретируемым вариантом.